In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os

# 设置中文字体（避免绘图乱码）
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 1. 读取数据

file_path = r'D:\数模协会\学术部\Examples.csv'
df = pd.read_csv(file_path, encoding='utf-8-sig')
print(f"数据集形状：{df.shape}")
print("\n前5行数据：")
print(df.head())

数据集形状：(33, 5)

前5行数据：
      学生ID  姓名     语文     数学     英语
0  2024001  张伟  112.0   87.0  105.0
1  2024002  王芳    NaN  137.0  108.0
2  2024003  李娜  114.0   99.0   98.0
3  2024004  赵磊    NaN   78.0   98.0
4  2024005  陈静  101.0  200.0  118.0


In [3]:
# 2. 数据概览
print("\n=== 数据基本信息 ===")
print(df.info())
print("\n=== 描述性统计 ===")
print(df.describe())


=== 数据基本信息 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   学生ID    33 non-null     int64  
 1   姓名      32 non-null     object 
 2   语文      27 non-null     float64
 3   数学      29 non-null     float64
 4   英语      31 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 1.4+ KB
None

=== 描述性统计 ===
               学生ID          语文          数学          英语
count  3.300000e+01   27.000000   29.000000   31.000000
mean   2.024014e+06   98.407407  102.413793  104.451613
std    9.264424e+00   23.482005   25.930563   23.219013
min    2.024001e+06   -5.000000   64.000000  -10.000000
25%    2.024006e+06   95.500000   87.000000  103.500000
50%    2.024014e+06  101.000000   99.000000  108.000000
75%    2.024022e+06  111.000000  112.000000  113.000000
max    2.024030e+06  128.000000  200.000000  125.000000


In [4]:
# 3. 重复值检测与剔除
print("\n=== 重复值检查 ===")
# 检测完全重复的行
duplicate_before = df.duplicated().sum() # duplicate_before = df.duplicated(subset=['学生ID', '姓名'], keep='last').sum()
print(f"重复行数量（剔除前）：{duplicate_before}")

# 显示重复的行
if duplicate_before > 0:
    print("\n重复行示例：")
    dup_rows = df[df.duplicated(keep=False)]  # dup_rows = df[df.duplicated(subset=['学生ID', '姓名'], keep=False)]
    print(dup_rows.sort_values(by='学生ID').head(10))

# 剔除重复行
df.drop_duplicates(inplace=True) # df.drop_duplicates(subset=['学生ID', '姓名'], keep='last', inplace=True)
print(f"\n重复行数量（剔除后）：{df.duplicated().sum()}")
print(f"剔除重复后数据集形状：{df.shape}")


=== 重复值检查 ===
重复行数量（剔除前）：3

重复行示例：
       学生ID  姓名     语文     数学     英语
0   2024001  张伟  112.0   87.0  105.0
30  2024001  张伟  112.0   87.0  105.0
1   2024002  王芳    NaN  137.0  108.0
31  2024002  王芳    NaN  137.0  108.0
2   2024003  李娜  114.0   99.0   98.0
32  2024003  李娜  114.0   99.0   98.0

重复行数量（剔除后）：0
剔除重复后数据集形状：(30, 5)


In [5]:
# 4. 异常值处理
# 定义成绩列的合理范围
lower_bound, upper_bound = 0, 150
score_columns = ['语文', '数学', '英语']

# 逐列检测异常值
for col in score_columns:
    # 找出异常值（小于0或大于150）
    outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    count = outlier_mask.sum()
    print(f"{col} 列异常值数量：{count}")
    if count > 0:
        print(f"  异常值详情：")
        print(df.loc[outlier_mask, ['学生ID', '姓名', col]])

# 将异常值转为空值 (NaN) 
df_cleaned = df.copy()
print("\n=== 处理方式：异常值替换为 NaN ===")

for col in score_columns:
    outlier_mask = (df_cleaned[col] < lower_bound) | (df_cleaned[col] > upper_bound)
    replace_count = outlier_mask.sum()
    if replace_count > 0:
        df_cleaned.loc[outlier_mask, col] = np.nan
        print(f"{col} 列已将 {replace_count} 个异常值替换为 NaN")

# 查看替换后的缺失情况
print("\n替换为 NaN 后的数据基本信息：")
print(df_cleaned.info())

"""
# ---------- 方案一：基于 IQR 箱线图检测异常，并替换为 NaN ----------
score_columns = ['语文', '数学', '英语']
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (data[column] < lower) | (data[column] > upper)
    
df_cleaned = df.copy()
print("\n=== 方案一：IQR 方法检测异常值 ===")
for col in score_columns:
    mask_iqr = detect_outliers_iqr(df, col)
    count = mask_iqr.sum()
    print(f"{col} 列 IQR 异常值数量：{count}")
    if count > 0:
        print(df.loc[mask_iqr, ['学生ID', '姓名', col]])
        # 将异常值替换为 NaN
        df_cleaned.loc[mask_iqr, col] = np.nan
        print(f"  已将 {count} 个异常值替换为 NaN")
print("\n替换为 NaN 后的数据基本信息：")
print(df_cleaned.info())
"""

"""
# ---------- 方案二：基于 3sigma 原则检测异常，并替换为 NaN ----------
score_columns = ['语文', '数学', '英语']
def detect_outliers_3sigma(data, column):
    mean = data[column].mean()
    std = data[column].std()
    lower = mean - 3 * std
    upper = mean + 3 * std
    return (data[column] < lower) | (data[column] > upper)
    
df_cleaned = df.copy()
print("\n=== 方案二：3sigma 方法检测异常值 ===")
for col in score_columns:
    mask_3sigma = detect_outliers_3sigma(df, col)
    count = mask_3sigma.sum()
    print(f"{col} 列 3sigma 异常值数量：{count}")
    if count > 0:
        print(df.loc[mask_3sigma, ['学生ID', '姓名', col]])
        # 将异常值替换为 NaN
        df_cleaned.loc[mask_3sigma, col] = np.nan
        print(f"  已将 {count} 个异常值替换为 NaN")
print("\n替换为 NaN 后的数据基本信息：")
print(df_cleaned.info())
"""
"""
# ---------- 备选处理方式：删除异常值所在行 ----------
# 生成一个布尔序列，标记至少有一科异常的行
outlier_rows = pd.Series(False, index=df.index)
for col in score_columns:
    outlier_rows |= ((df[col] < lower_bound) | (df[col] > upper_bound))

df_dropped = df[~outlier_rows].copy()
print(f"删除异常行前数据量：{len(df)}，删除后：{len(df_dropped)}")
"""

语文 列异常值数量：1
  异常值详情：
       学生ID  姓名   语文
19  2024020  沈梦 -5.0
数学 列异常值数量：1
  异常值详情：
      学生ID  姓名     数学
4  2024005  陈静  200.0
英语 列异常值数量：1
  异常值详情：
       学生ID  姓名    英语
11  2024012  胡强 -10.0

=== 处理方式：异常值替换为 NaN ===
语文 列已将 1 个异常值替换为 NaN
数学 列已将 1 个异常值替换为 NaN
英语 列已将 1 个异常值替换为 NaN

替换为 NaN 后的数据基本信息：
<class 'pandas.core.frame.DataFrame'>
Index: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   学生ID    30 non-null     int64  
 1   姓名      29 non-null     object 
 2   语文      24 non-null     float64
 3   数学      25 non-null     float64
 4   英语      27 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 1.4+ KB
None


'\n# ---------- 备选处理方式：删除异常值所在行 ----------\n# 生成一个布尔序列，标记至少有一科异常的行\noutlier_rows = pd.Series(False, index=df.index)\nfor col in score_columns:\n    outlier_rows |= ((df[col] < lower_bound) | (df[col] > upper_bound))\n\ndf_dropped = df[~outlier_rows].copy()\nprint(f"删除异常行前数据量：{len(df)}，删除后：{len(df_dropped)}")\n'

In [6]:
# 5. 缺失值处理
print("处理前缺失值数量：")
print(df_cleaned[score_columns + ['姓名']].isnull().sum())

# 数值型列用中位数填充
for col in score_columns:
    median_val = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(median_val)
    print(f"{col} 列用中位数 {median_val:.2f} 填充")

# 姓名列先将空字符串转为 NaN，再用“未知”填充
df_cleaned['姓名'] = df_cleaned['姓名'].replace('', np.nan).fillna('未知')
print("姓名列缺失值已填充为'未知'")

# 保存最终清洗后的数据
cleaned_file = r'D:\数模协会\学术部\result.csv'
df_cleaned.to_csv(cleaned_file, index=False, encoding='utf-8-sig')
print(f"\n最终清洗后数据已保存至：{cleaned_file}")

处理前缺失值数量：
语文    6
数学    5
英语    3
姓名    1
dtype: int64
语文 列用中位数 101.00 填充
数学 列用中位数 97.00 填充
英语 列用中位数 109.00 填充
姓名列缺失值已填充为'未知'

最终清洗后数据已保存至：D:\数模协会\学术部\result.csv
